# PubMedQA Fine-tuning Demo

This notebook demonstrates fine-tuning Qwen3-0.6B on the PubMedQA dataset using:
1. **SFT** (Supervised Fine-Tuning)
2. **GRPO** with answer correctness reward
3. **GRPO** with embedding-based distance reward
4. **GRPO curriculum**: format reward then embedding reward
5. **GRPO curriculum on SFT**: GRPO fine-tuning on top of the SFT model

We then evaluate all checkpoints on the test set and compare methods.

In [ ]:
import os
import re
import time
import glob
import json
from typing import Any
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from peft import LoraConfig, PeftModel
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoProcessor, AutoModel, AutoTokenizer
from trl import GRPOTrainer, GRPOConfig, SFTTrainer, SFTConfig
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import StaticEmbedding

## 0. Common constants and utility functions

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
DATASET_NAME = "bigbio/pubmed_qa"
OUTPUT_DIR_ROOT = "logs"
REWARD_EMBEDDING_MODEL = "NeuML/pubmedbert-base-embeddings-8M"
LAMBDA = 0.001
MAX_COMPLETION_LENGTH = 1024
SAVE_STEPS = 20
SEED = 42

torch.manual_seed(SEED)

In [ ]:
# ──────────────────────────────────────────────
# Prompt formatting and answer extraction
# ──────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a medical expert. Given a clinical research abstract, "
    "answer the yes/no question and provide a one-sentence explanation."
)


def format_prompt(sample: dict) -> str:
    contexts = "\n\n".join(
        f"[{label}]\n{context}"
        for label, context in zip(sample["LABELS"], sample["CONTEXTS"])
    )
    prompt = f"""<question>
{sample["QUESTION"]}
</question>

<abstract>
{contexts}
</abstract>

Answer the question with exactly "yes" or "no", then provide a one-sentence explanation of your reasoning.

Respond in this exact format:
<answer>yes/no</answer>
<long_answer>One sentence explanation here.</long_answer>"""
    return prompt


def extract_answer(text: str) -> str | None:
    match = re.search(r"<answer>\s*(yes|no)\s*</answer>", text, re.IGNORECASE)
    return match.group(1).strip().lower() if match else None


def extract_long_answer(text: str) -> str | None:
    match = re.search(r"<long_answer>(.*?)</long_answer>", text, re.DOTALL)
    return match.group(1).strip() if match else None

In [ ]:
# ──────────────────────────────────────────────
# Reward functions
# ──────────────────────────────────────────────

def answer_correctness_reward(
    completions: list[list[dict[str, str]]],
    answer: list[str],
    **kwargs: Any,
) -> list[float]:
    """Returns 1.0 if the extracted <answer> matches the ground truth, else 0.0."""
    rewards = []
    for completion, ground_truth in zip(completions, answer):
        text = completion[0]["content"]
        predicted = extract_answer(text)
        reward = (
            1.0
            if (predicted is not None and predicted == ground_truth.strip().lower())
            else 0.0
        )
        rewards.append(reward)
    return rewards


def format_compliance_reward(
    completions: list[list[dict[str, str]]],
    **kwargs: Any,
) -> list[float]:
    """Returns 1.0 if both <answer> and <long_answer> tags are present and non-empty, else 0.0."""
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        has_answer = extract_answer(text) is not None
        has_long_answer = extract_long_answer(text) is not None
        rewards.append(1.0 if (has_answer and has_long_answer) else 0.0)
    return rewards

In [ ]:
# ──────────────────────────────────────────────
# Embedding-based reward (SentenceTransformer)
# ──────────────────────────────────────────────

class SentenceTransformerMahalanobisReward(nn.Module):
    def __init__(
        self,
        model_name: str = "google/embeddinggemma-300m",
        train_encoder: bool = False,
        train_matrix: bool = True,
        max_length: int = 2048,
    ):
        super().__init__()
        self.max_length = max_length
        if model_name == "NeuML/pubmedbert-base-embeddings-8M":
            static = StaticEmbedding.from_model2vec(model_name)
            self.model = SentenceTransformer(modules=[static])
        else:
            self.model = SentenceTransformer(model_name, trust_remote_code=True)
            self.model.max_seq_length = max_length

        if not train_encoder:
            for p in self.model.parameters():
                p.requires_grad = False

        hidden_size = self.model.get_sentence_embedding_dimension()
        self.matrix = nn.Parameter(torch.eye(hidden_size), requires_grad=train_matrix)

    def _device(self):
        return self.matrix.device

    def set_matrix(self, matrix: torch.Tensor, trainable: bool = False):
        hidden = self.model.get_sentence_embedding_dimension()
        assert matrix.shape == (hidden, hidden)
        self.matrix = nn.Parameter(matrix.to(self._device()), requires_grad=trainable)

    def encode(self, y, y_hat):
        if isinstance(y, str):
            y = [y]
        if isinstance(y_hat, str):
            y_hat = [y_hat]
        e_y = self.model.encode(y, convert_to_tensor=True, device=self._device(), show_progress_bar=False)
        e_y_hat = self.model.encode(y_hat, convert_to_tensor=True, device=self._device(), show_progress_bar=False)
        return e_y, e_y_hat

    def forward(self, y, y_hat):
        e_y, e_y_hat = self.encode(y, y_hat)
        diff = e_y - e_y_hat
        return -torch.einsum("...i,ij,...j->...", diff, self.matrix, diff)


def embedding_reward_func_constructor_pubmedqa(
    model: str,
    U_star=None,
    pooling: str = "mean",
    verbose: int = 0,
    max_length: int = 2048,
    answer_only: bool = True,
    sentence_transformer: bool = False,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if sentence_transformer:
        embedder = SentenceTransformerMahalanobisReward(
            model_name=model,
            train_encoder=False,
            train_matrix=False,
            max_length=max_length,
        )
        min_reward = -100.0
    else:
        # For non-sentence-transformer models, use AutoModel-based embedder
        # (not shown here for brevity; use sentence_transformer=True)
        raise NotImplementedError("Set sentence_transformer=True for this demo")

    if U_star is not None:
        embedder.set_matrix(matrix=nn.Parameter(U_star))
    embedder.to(device)

    def reward_func(
        completions: list[list[dict[str, str]]],
        answer: list[str],
        long_answer: list[str],
        **kwargs,
    ) -> list[float]:
        rewards = []
        for completion, a, la in zip(completions, answer, long_answer):
            try:
                text = completion[0]["content"]
                predicted_answer = extract_answer(text)
                predicted_long = extract_long_answer(text)

                if predicted_answer is None:
                    rewards.append(min_reward)
                    continue

                if answer_only:
                    y_hat = predicted_answer
                    y = a.strip().lower()
                else:
                    if predicted_long is None:
                        rewards.append(min_reward)
                        continue
                    y_hat = predicted_long
                    y = la.strip()

                reward = embedder(y_hat=y_hat, y=y)
                rewards.append(reward.item())

            except Exception as e:
                if verbose > 0:
                    print(f"completion: {completion}\nanswer: {a}\nlong_answer: {la}\nError: {e}")
                rewards.append(min_reward)

        return rewards

    return reward_func

In [ ]:
# ──────────────────────────────────────────────
# Dataset preparation helpers
# ──────────────────────────────────────────────

def prepare_dataset(hf_dataset) -> Dataset:
    """Map raw HuggingFace dataset rows to prompt/answer pairs for GRPO."""
    def process(sample):
        return {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": format_prompt(sample)},
            ],
            "answer": sample["final_decision"],
            "long_answer": sample["LONG_ANSWER"],
        }
    return hf_dataset.map(process, remove_columns=hf_dataset.column_names)


def format_sft_sample(sample: dict) -> dict:
    contexts = "\n\n".join(
        f"[{label}]\n{context}"
        for label, context in zip(sample["LABELS"], sample["CONTEXTS"])
    )
    user_content = f"""<question>
{sample["QUESTION"]}
</question>

<abstract>
{contexts}
</abstract>

Answer with "yes" or "no", then provide a one-sentence explanation.

Respond in this exact format:
<answer>yes/no</answer>
<long_answer>One sentence explanation here.</long_answer>"""

    assistant_content = (
        f"<think>\n{sample['LONG_ANSWER']}\n</think>\n"
        f"<answer>{sample['final_decision']}</answer>\n"
        f"<long_answer>{sample['LONG_ANSWER']}</long_answer>"
    )

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        "completion": [
            {"role": "assistant", "content": assistant_content},
        ],
    }


def prepare_sft_dataset(hf_dataset, num_samples: int | None = None):
    if num_samples is not None:
        hf_dataset = hf_dataset.select(range(num_samples))
    hf_dataset = hf_dataset.filter(
        lambda x: x["final_decision"] is not None and x["LONG_ANSWER"] is not None
    )
    return hf_dataset.map(
        format_sft_sample,
        remove_columns=hf_dataset.column_names,
    )

## 1. Load and explore the dataset

In [ ]:
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print(f"\nTrain split size: {len(raw_dataset['train'])}")
print(f"Columns: {raw_dataset['train'].column_names}")

In [ ]:
# Inspect a single example
example = raw_dataset["train"][0]
for key, value in example.items():
    print(f"--- {key} ---")
    if isinstance(value, list) and len(value) > 3:
        print(value[:3], "...")
    else:
        print(value)
    print()

In [ ]:
# Distribution of final_decision labels
from collections import Counter

labels = raw_dataset["train"]["final_decision"]
label_counts = Counter(labels)
print("Label distribution:")
for label, count in label_counts.most_common():
    print(f"  {label}: {count} ({100 * count / len(labels):.1f}%)")

In [ ]:
# Preview the formatted prompt
print("=== Formatted prompt ===")
print(format_prompt(raw_dataset["train"][0]))

## 2. Prepare train/test splits

In [ ]:
# GRPO dataset
grpo_dataset = prepare_dataset(raw_dataset["train"])
grpo_split = grpo_dataset.train_test_split(test_size=0.1, seed=SEED)
grpo_train = grpo_split["train"]
grpo_test = grpo_split["test"]
print(f"GRPO train: {len(grpo_train)}, test: {len(grpo_test)}")
print(f"Example: {grpo_train[0]}")

In [ ]:
# SFT dataset
sft_dataset = prepare_sft_dataset(raw_dataset["train"])
sft_split = sft_dataset.train_test_split(test_size=0.1, seed=SEED)
sft_train = sft_split["train"]
sft_test = sft_split["test"]
print(f"SFT train: {len(sft_train)}, test: {len(sft_test)}")
print(f"Example: {sft_train[0]}")

## 3. SFT Fine-tuning

In [ ]:
# Load model and tokenizer
sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    dtype="bfloat16",
    use_cache=True,
    device_map="auto",
)
sft_tokenizer = AutoProcessor.from_pretrained(MODEL_NAME, padding_side="left")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
sft_output_dir = f"{OUTPUT_DIR_ROOT}/{MODEL_NAME.split('/')[-1]}/{DATASET_NAME.split('/')[-1]}/[peft]trl-sft-{timestamp}"
os.makedirs(sft_output_dir, exist_ok=True)

sft_peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["k_proj", "q_proj", "v_proj"],
)

sft_training_args = SFTConfig(
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=5e-5,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    report_to="tensorboard",
    save_steps=SAVE_STEPS,
    output_dir=sft_output_dir,
    max_length=2048,
    gradient_checkpointing=False,
    push_to_hub=False,
    completion_only_loss=True,
    packing=False,
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_training_args,
    train_dataset=sft_train,
    peft_config=sft_peft_config,
)

print(f"SFT output dir: {sft_output_dir}")

In [ ]:
# Train SFT
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

torch.cuda.synchronize()
t0 = time.time()

sft_trainer_stats = sft_trainer.train()

torch.cuda.synchronize()
t1 = time.time()
print(f"Total SFT training time: {round(t1 - t0, 2)} seconds.")

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Peak reserved memory = {used_memory} GB.")

# Save final model
sft_trainer.save_model(sft_output_dir)

In [ ]:
# Clean up SFT model to free GPU memory
del sft_model, sft_trainer
torch.cuda.empty_cache()

## 4. GRPO Fine-tuning: Answer Correctness Reward

In [ ]:
# Load fresh model
grpo_ac_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    dtype="bfloat16",
    use_cache=True,
    device_map="auto",
)
grpo_ac_tokenizer = AutoProcessor.from_pretrained(MODEL_NAME, padding_side="left")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
grpo_ac_output_dir = f"{OUTPUT_DIR_ROOT}/{MODEL_NAME.split('/')[-1]}/{DATASET_NAME.split('/')[-1]}/grpo-answer-correctness-{timestamp}"
os.makedirs(grpo_ac_output_dir, exist_ok=True)

grpo_ac_peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

# Reward: format compliance (weight 0.5) + answer correctness (weight 0.5)
grpo_ac_reward_funcs = [format_compliance_reward, answer_correctness_reward]
grpo_ac_reward_weights = [0.5, 0.5]

grpo_ac_training_args = GRPOConfig(
    do_eval=False,
    learning_rate=5e-5,
    num_train_epochs=1,
    max_steps=400,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=256,
    num_generations=8,
    beta=LAMBDA,
    gradient_checkpointing=False,
    fp16=False,
    bf16=True,
    output_dir=grpo_ac_output_dir,
    logging_steps=1,
    save_steps=SAVE_STEPS,
    report_to="tensorboard",
    push_to_hub=False,
    log_completions=True,
    reward_weights=grpo_ac_reward_weights,
)

grpo_ac_trainer = GRPOTrainer(
    model=grpo_ac_model,
    processing_class=grpo_ac_tokenizer,
    reward_funcs=grpo_ac_reward_funcs,
    args=grpo_ac_training_args,
    train_dataset=grpo_train,
    peft_config=grpo_ac_peft_config,
)

print(f"GRPO (answer correctness) output dir: {grpo_ac_output_dir}")

In [ ]:
# Train
torch.cuda.synchronize()
t0 = time.time()

grpo_ac_trainer.train()

torch.cuda.synchronize()
t1 = time.time()
print(f"Total GRPO (answer correctness) training time: {round(t1 - t0, 2)} seconds.")

grpo_ac_trainer.save_model(grpo_ac_output_dir)

In [ ]:
del grpo_ac_model, grpo_ac_trainer
torch.cuda.empty_cache()

## 5. GRPO Fine-tuning: Embedding Distance Reward

In [ ]:
# Load fresh model
grpo_emb_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    dtype="bfloat16",
    use_cache=True,
    device_map="auto",
)
grpo_emb_tokenizer = AutoProcessor.from_pretrained(MODEL_NAME, padding_side="left")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
grpo_emb_output_dir = f"{OUTPUT_DIR_ROOT}/{MODEL_NAME.split('/')[-1]}/{DATASET_NAME.split('/')[-1]}/grpo-embedding-{timestamp}"
os.makedirs(grpo_emb_output_dir, exist_ok=True)

grpo_emb_peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

# Build embedding reward function
embedding_reward_fn = embedding_reward_func_constructor_pubmedqa(
    model=REWARD_EMBEDDING_MODEL,
    U_star=None,
    pooling="cls",
    max_length=MAX_COMPLETION_LENGTH + 256,
    answer_only=False,
    sentence_transformer=True,
)

FORMAT_WEIGHT = 0.5
grpo_emb_reward_funcs = [format_compliance_reward, embedding_reward_fn]
grpo_emb_reward_weights = [FORMAT_WEIGHT, 1.0 - FORMAT_WEIGHT]

grpo_emb_training_args = GRPOConfig(
    do_eval=False,
    learning_rate=5e-5,
    num_train_epochs=1,
    max_steps=400,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=256,
    num_generations=8,
    beta=LAMBDA,
    gradient_checkpointing=False,
    fp16=False,
    bf16=True,
    output_dir=grpo_emb_output_dir,
    logging_steps=1,
    save_steps=SAVE_STEPS,
    report_to="tensorboard",
    push_to_hub=False,
    log_completions=True,
    reward_weights=grpo_emb_reward_weights,
)

grpo_emb_trainer = GRPOTrainer(
    model=grpo_emb_model,
    processing_class=grpo_emb_tokenizer,
    reward_funcs=grpo_emb_reward_funcs,
    args=grpo_emb_training_args,
    train_dataset=grpo_train,
    peft_config=grpo_emb_peft_config,
)

print(f"GRPO (embedding) output dir: {grpo_emb_output_dir}")

In [ ]:
torch.cuda.synchronize()
t0 = time.time()

grpo_emb_trainer.train()

torch.cuda.synchronize()
t1 = time.time()
print(f"Total GRPO (embedding) training time: {round(t1 - t0, 2)} seconds.")

grpo_emb_trainer.save_model(grpo_emb_output_dir)

In [ ]:
del grpo_emb_model, grpo_emb_trainer, embedding_reward_fn
torch.cuda.empty_cache()

## 6. GRPO Curriculum: Format Reward then Embedding Reward

In [ ]:
# Load fresh model
grpo_cur_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    dtype="bfloat16",
    use_cache=True,
    device_map="auto",
)
grpo_cur_tokenizer = AutoProcessor.from_pretrained(MODEL_NAME, padding_side="left")

In [ ]:
# ── Phase 1: Format reward only ──
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
grpo_cur_output_dir = f"{OUTPUT_DIR_ROOT}/{MODEL_NAME.split('/')[-1]}/{DATASET_NAME.split('/')[-1]}/grpo-curriculum-{timestamp}"
grpo_cur_phase1_dir = f"{grpo_cur_output_dir}/phase1-format"
os.makedirs(grpo_cur_phase1_dir, exist_ok=True)

grpo_cur_peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

grpo_cur_phase1_args = GRPOConfig(
    do_eval=False,
    learning_rate=5e-5,
    num_train_epochs=1,
    max_steps=200,  # Half the steps for phase 1
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=256,
    num_generations=8,
    beta=LAMBDA,
    gradient_checkpointing=False,
    fp16=False,
    bf16=True,
    output_dir=grpo_cur_phase1_dir,
    logging_steps=1,
    save_steps=SAVE_STEPS,
    report_to="tensorboard",
    push_to_hub=False,
    log_completions=True,
    reward_weights=[1.0],
)

grpo_cur_phase1_trainer = GRPOTrainer(
    model=grpo_cur_model,
    processing_class=grpo_cur_tokenizer,
    reward_funcs=[format_compliance_reward],
    args=grpo_cur_phase1_args,
    train_dataset=grpo_train,
    peft_config=grpo_cur_peft_config,
)

print(f"Curriculum phase 1 output dir: {grpo_cur_phase1_dir}")

In [ ]:
# Train phase 1
torch.cuda.synchronize()
t0 = time.time()

grpo_cur_phase1_trainer.train()

torch.cuda.synchronize()
t1 = time.time()
print(f"Curriculum phase 1 training time: {round(t1 - t0, 2)} seconds.")

grpo_cur_phase1_trainer.save_model(grpo_cur_phase1_dir)

In [ ]:
del grpo_cur_phase1_trainer
torch.cuda.empty_cache()

In [ ]:
# ── Phase 2: Embedding reward on the phase-1 model ──
grpo_cur_phase2_dir = f"{grpo_cur_output_dir}/phase2-embedding"
os.makedirs(grpo_cur_phase2_dir, exist_ok=True)

# Load phase-1 model (base + LoRA adapter)
grpo_cur_phase2_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    dtype="bfloat16",
    use_cache=True,
    device_map="auto",
)
grpo_cur_phase2_model = PeftModel.from_pretrained(grpo_cur_phase2_model, grpo_cur_phase1_dir)
grpo_cur_phase2_model = grpo_cur_phase2_model.merge_and_unload()
grpo_cur_phase2_tokenizer = AutoProcessor.from_pretrained(MODEL_NAME, padding_side="left")

embedding_reward_fn_cur = embedding_reward_func_constructor_pubmedqa(
    model=REWARD_EMBEDDING_MODEL,
    U_star=None,
    pooling="cls",
    max_length=MAX_COMPLETION_LENGTH + 256,
    answer_only=False,
    sentence_transformer=True,
)

grpo_cur_phase2_peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

FORMAT_WEIGHT = 0.5
grpo_cur_phase2_args = GRPOConfig(
    do_eval=False,
    learning_rate=5e-5,
    num_train_epochs=1,
    max_steps=200,  # Remaining half of steps
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=256,
    num_generations=8,
    beta=LAMBDA,
    gradient_checkpointing=False,
    fp16=False,
    bf16=True,
    output_dir=grpo_cur_phase2_dir,
    logging_steps=1,
    save_steps=SAVE_STEPS,
    report_to="tensorboard",
    push_to_hub=False,
    log_completions=True,
    reward_weights=[FORMAT_WEIGHT, 1.0 - FORMAT_WEIGHT],
)

grpo_cur_phase2_trainer = GRPOTrainer(
    model=grpo_cur_phase2_model,
    processing_class=grpo_cur_phase2_tokenizer,
    reward_funcs=[format_compliance_reward, embedding_reward_fn_cur],
    args=grpo_cur_phase2_args,
    train_dataset=grpo_train,
    peft_config=grpo_cur_phase2_peft_config,
)

print(f"Curriculum phase 2 output dir: {grpo_cur_phase2_dir}")

In [ ]:
torch.cuda.synchronize()
t0 = time.time()

grpo_cur_phase2_trainer.train()

torch.cuda.synchronize()
t1 = time.time()
print(f"Curriculum phase 2 training time: {round(t1 - t0, 2)} seconds.")

grpo_cur_phase2_trainer.save_model(grpo_cur_phase2_dir)

In [ ]:
del grpo_cur_model, grpo_cur_phase2_model, grpo_cur_phase2_trainer, embedding_reward_fn_cur
torch.cuda.empty_cache()

## 7. GRPO Curriculum on SFT: SFT then GRPO

In [ ]:
# Load the SFT-finetuned model and continue with GRPO
grpo_sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    dtype="bfloat16",
    use_cache=True,
    device_map="auto",
)
grpo_sft_model = PeftModel.from_pretrained(grpo_sft_model, sft_output_dir)
grpo_sft_model = grpo_sft_model.merge_and_unload()
grpo_sft_tokenizer = AutoProcessor.from_pretrained(MODEL_NAME, padding_side="left")

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
grpo_sft_output_dir = f"{OUTPUT_DIR_ROOT}/{MODEL_NAME.split('/')[-1]}/{DATASET_NAME.split('/')[-1]}/grpo-on-sft-{timestamp}"
os.makedirs(grpo_sft_output_dir, exist_ok=True)

grpo_sft_peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

embedding_reward_fn_sft = embedding_reward_func_constructor_pubmedqa(
    model=REWARD_EMBEDDING_MODEL,
    U_star=None,
    pooling="cls",
    max_length=MAX_COMPLETION_LENGTH + 256,
    answer_only=False,
    sentence_transformer=True,
)

FORMAT_WEIGHT = 0.5
grpo_sft_reward_funcs = [format_compliance_reward, embedding_reward_fn_sft]
grpo_sft_reward_weights = [FORMAT_WEIGHT, 1.0 - FORMAT_WEIGHT]

grpo_sft_training_args = GRPOConfig(
    do_eval=False,
    learning_rate=5e-5,
    num_train_epochs=1,
    max_steps=400,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=256,
    num_generations=8,
    beta=LAMBDA,
    gradient_checkpointing=False,
    fp16=False,
    bf16=True,
    output_dir=grpo_sft_output_dir,
    logging_steps=1,
    save_steps=SAVE_STEPS,
    report_to="tensorboard",
    push_to_hub=False,
    log_completions=True,
    reward_weights=grpo_sft_reward_weights,
)

grpo_sft_trainer = GRPOTrainer(
    model=grpo_sft_model,
    processing_class=grpo_sft_tokenizer,
    reward_funcs=grpo_sft_reward_funcs,
    args=grpo_sft_training_args,
    train_dataset=grpo_train,
    peft_config=grpo_sft_peft_config,
)

print(f"GRPO on SFT output dir: {grpo_sft_output_dir}")

In [ ]:
torch.cuda.synchronize()
t0 = time.time()

grpo_sft_trainer.train()

torch.cuda.synchronize()
t1 = time.time()
print(f"Total GRPO-on-SFT training time: {round(t1 - t0, 2)} seconds.")

grpo_sft_trainer.save_model(grpo_sft_output_dir)

In [ ]:
del grpo_sft_model, grpo_sft_trainer, embedding_reward_fn_sft
torch.cuda.empty_cache()

## 8. Evaluation: Answer Correctness on Test Set

For each method, we load every saved checkpoint, generate completions on the test set,
and compute the answer correctness accuracy.

In [ ]:
def find_checkpoints(output_dir: str) -> list[str]:
    """Find all checkpoint directories sorted by step number."""
    checkpoint_dirs = sorted(
        glob.glob(os.path.join(output_dir, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1]),
    )
    # Also include the final model directory itself
    checkpoint_dirs.append(output_dir)
    return checkpoint_dirs


def get_step_number(checkpoint_path: str) -> int:
    """Extract step number from checkpoint path."""
    basename = os.path.basename(checkpoint_path)
    if basename.startswith("checkpoint-"):
        return int(basename.split("-")[-1])
    return -1  # Final model


def evaluate_checkpoint(
    checkpoint_path: str,
    test_dataset,
    base_model_name: str = MODEL_NAME,
    max_new_tokens: int = MAX_COMPLETION_LENGTH,
    batch_size: int = 8,
) -> float:
    """Load a checkpoint and evaluate answer correctness on the test set."""
    # Load base model + LoRA adapter
    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        attn_implementation="flash_attention_2",
        dtype="bfloat16",
        device_map="auto",
    )
    # Try loading as LoRA adapter; if it fails, it might be a merged model
    try:
        model = PeftModel.from_pretrained(model, checkpoint_path)
        model = model.merge_and_unload()
    except Exception:
        # If not a PEFT checkpoint, load directly
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint_path,
            attn_implementation="flash_attention_2",
            dtype="bfloat16",
            device_map="auto",
        )

    tokenizer = AutoProcessor.from_pretrained(base_model_name, padding_side="left")
    model.eval()

    correct = 0
    total = 0

    for i in range(0, len(test_dataset), batch_size):
        batch = test_dataset[i : i + batch_size]
        prompts = batch["prompt"]
        answers = batch["answer"]

        # Apply chat template to each prompt
        texts = [
            tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
            for p in prompts
        ]
        inputs = tokenizer(
            texts, return_tensors="pt", padding=True, truncation=True, max_length=256
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )

        # Decode only the generated tokens
        for j, (output, gt) in enumerate(zip(outputs, answers)):
            generated = tokenizer.decode(
                output[inputs["input_ids"].shape[1] :], skip_special_tokens=True
            )
            predicted = extract_answer(generated)
            if predicted is not None and predicted == gt.strip().lower():
                correct += 1
            total += 1

    del model
    torch.cuda.empty_cache()

    return correct / total if total > 0 else 0.0

In [ ]:
# Evaluate base model (no fine-tuning)
print("Evaluating base model...")
base_model_accuracy = evaluate_checkpoint(
    checkpoint_path=MODEL_NAME,  # Will fail PEFT load and fall back to direct load
    test_dataset=grpo_test,
    base_model_name=MODEL_NAME,
)
print(f"Base model accuracy: {base_model_accuracy:.4f}")

In [ ]:
# Define all methods and their output directories
methods = {
    "SFT": sft_output_dir,
    "GRPO (Answer Correctness)": grpo_ac_output_dir,
    "GRPO (Embedding)": grpo_emb_output_dir,
    "GRPO Curriculum (Format->Embed)": [grpo_cur_phase1_dir, grpo_cur_phase2_dir],
    "GRPO on SFT": grpo_sft_output_dir,
}

# Evaluate all checkpoints for each method
results = {}  # method_name -> {steps: [...], accuracies: [...]}

for method_name, output_dir in methods.items():
    print(f"\nEvaluating: {method_name}")

    # Handle curriculum methods with two phase directories
    if isinstance(output_dir, list):
        all_checkpoints = []
        step_offset = 0
        for phase_dir in output_dir:
            phase_checkpoints = find_checkpoints(phase_dir)
            for cp in phase_checkpoints:
                step = get_step_number(cp)
                if step == -1:
                    # Final model of this phase: use max_steps as step
                    step = step_offset + 200  # Each phase is 200 steps
                else:
                    step = step_offset + step
                all_checkpoints.append((step, cp))
            step_offset += 200
    else:
        all_checkpoints = []
        for cp in find_checkpoints(output_dir):
            step = get_step_number(cp)
            if step == -1:
                step = 400  # max_steps
            all_checkpoints.append((step, cp))

    steps = []
    accuracies = []
    for step, cp in sorted(all_checkpoints, key=lambda x: x[0]):
        print(f"  Step {step}: {cp}")
        acc = evaluate_checkpoint(cp, grpo_test)
        print(f"    Accuracy: {acc:.4f}")
        steps.append(step)
        accuracies.append(acc)

    results[method_name] = {"steps": steps, "accuracies": accuracies}

In [ ]:
# Plot answer correctness comparison
plt.figure(figsize=(12, 6))

for method_name, data in results.items():
    plt.plot(data["steps"], data["accuracies"], marker="o", label=method_name)

# Add base model as a horizontal reference
plt.axhline(y=base_model_accuracy, color="gray", linestyle="--", alpha=0.7, label="Base model")
plt.scatter([0], [base_model_accuracy], color="gray", s=100, zorder=5, marker="D")

plt.xlabel("Training Steps")
plt.ylabel("Test Accuracy (Answer Correctness)")
plt.title("PubMedQA Fine-tuning Comparison: Answer Correctness")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("pubmedqa_answer_correctness.png", dpi=150)
plt.show()

## 9. Evaluation: lm-eval-harness (MMLU & MedMCQA)

We use the `lm-eval` CLI to evaluate each checkpoint on MMLU and MedMCQA benchmarks.

In [ ]:
import subprocess

def run_lm_eval(model_path: str, tasks: str = "mmlu,medmcqa", output_path: str = "lm_eval_results") -> dict:
    """Run lm-eval-harness on a model checkpoint and return parsed results."""
    result_dir = os.path.join(output_path, os.path.basename(model_path))
    os.makedirs(result_dir, exist_ok=True)

    # Check if this is a PEFT adapter or a full model
    adapter_config = os.path.join(model_path, "adapter_config.json")
    if os.path.exists(adapter_config):
        # PEFT adapter: use peft with base model
        model_args = f"pretrained={MODEL_NAME},peft={model_path},dtype=bfloat16"
    else:
        model_args = f"pretrained={model_path},dtype=bfloat16"

    cmd = [
        "lm_eval",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", tasks,
        "--batch_size", "auto",
        "--output_path", result_dir,
        "--log_samples",
    ]

    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
    if result.returncode != 0:
        print(f"STDERR: {result.stderr[-500:]}")

    # Parse results from JSON output
    scores = {}
    for result_file in glob.glob(os.path.join(result_dir, "**", "results.json"), recursive=True):
        with open(result_file) as f:
            data = json.load(f)
        for task_name, task_results in data.get("results", {}).items():
            # Use acc_norm if available, otherwise acc
            acc = task_results.get("acc_norm,none", task_results.get("acc,none", None))
            if acc is not None:
                scores[task_name] = acc

    return scores

In [ ]:
# Evaluate base model
print("=== Evaluating base model on MMLU & MedMCQA ===")
base_benchmark_scores = run_lm_eval(MODEL_NAME)
print(f"Base model scores: {base_benchmark_scores}")

In [ ]:
# Evaluate all checkpoints on MMLU & MedMCQA
benchmark_results = {}  # method_name -> {steps: [...], mmlu: [...], medmcqa: [...]}

for method_name, output_dir in methods.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {method_name}")
    print(f"{'='*60}")

    if isinstance(output_dir, list):
        all_checkpoints = []
        step_offset = 0
        for phase_dir in output_dir:
            phase_checkpoints = find_checkpoints(phase_dir)
            for cp in phase_checkpoints:
                step = get_step_number(cp)
                if step == -1:
                    step = step_offset + 200
                else:
                    step = step_offset + step
                all_checkpoints.append((step, cp))
            step_offset += 200
    else:
        all_checkpoints = []
        for cp in find_checkpoints(output_dir):
            step = get_step_number(cp)
            if step == -1:
                step = 400
            all_checkpoints.append((step, cp))

    steps = []
    mmlu_scores = []
    medmcqa_scores = []

    for step, cp in sorted(all_checkpoints, key=lambda x: x[0]):
        print(f"\n  Step {step}: {cp}")
        scores = run_lm_eval(cp)
        print(f"    Scores: {scores}")

        steps.append(step)
        mmlu_scores.append(scores.get("mmlu", 0.0))
        medmcqa_scores.append(scores.get("medmcqa", 0.0))

    benchmark_results[method_name] = {
        "steps": steps,
        "mmlu": mmlu_scores,
        "medmcqa": medmcqa_scores,
    }

In [ ]:
# Plot MMLU comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# MMLU
ax = axes[0]
for method_name, data in benchmark_results.items():
    ax.plot(data["steps"], data["mmlu"], marker="o", label=method_name)

base_mmlu = base_benchmark_scores.get("mmlu", 0.0)
ax.axhline(y=base_mmlu, color="gray", linestyle="--", alpha=0.7, label="Base model")
ax.scatter([0], [base_mmlu], color="gray", s=100, zorder=5, marker="D")
ax.set_xlabel("Training Steps")
ax.set_ylabel("MMLU Score")
ax.set_title("MMLU Benchmark")
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)

# MedMCQA
ax = axes[1]
for method_name, data in benchmark_results.items():
    ax.plot(data["steps"], data["medmcqa"], marker="o", label=method_name)

base_medmcqa = base_benchmark_scores.get("medmcqa", 0.0)
ax.axhline(y=base_medmcqa, color="gray", linestyle="--", alpha=0.7, label="Base model")
ax.scatter([0], [base_medmcqa], color="gray", s=100, zorder=5, marker="D")
ax.set_xlabel("Training Steps")
ax.set_ylabel("MedMCQA Score")
ax.set_title("MedMCQA Benchmark")
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("PubMedQA Fine-tuning: Benchmark Comparison", fontsize=14)
plt.tight_layout()
plt.savefig("pubmedqa_benchmarks.png", dpi=150)
plt.show()

In [ ]:
# Save all results to JSON for later analysis
all_results = {
    "base_model_accuracy": base_model_accuracy,
    "base_benchmark_scores": base_benchmark_scores,
    "answer_correctness": results,
    "benchmarks": benchmark_results,
    "methods_dirs": {k: v if isinstance(v, str) else v for k, v in methods.items()},
}
with open("pubmedqa_demo_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Results saved to pubmedqa_demo_results.json")